In [1]:
import pandas as pd 
import os 
import sys

sys.path.append(os.path.abspath(os.path.join(os.getcwd() , os.pardir)))

from utils.helpers import get_db_engine
from utils.loggers_config import logger

engine = get_db_engine()

query = "SELECT * FROM engineered_churn_data"

try:
    logger.info("Attempting to fetch data from PostgreSQL...")
    df = pd.read_sql(query, engine)
    logger.info(f"Data ingestion successful! Loaded {df.shape[0]} rows and {df.shape[1]} columns.")
except Exception as e:
    logger.error(f"Failed to ingest data. Error: {str(e)}")
    raise e

df.head()

[2026-05-13 20:37:38,008: INFO: 4007626463: Attempting to fetch data from PostgreSQL...]
[2026-05-13 20:37:40,940: INFO: 4007626463: Data ingestion successful! Loaded 440832 rows and 18 columns.]


,Age,Tenure,Usage Frequency,Support Calls,Payment Delay,Contract Length,Total Spend,Last Interaction,Churn,Gender_male,Subscription Type_premium,Subscription Type_standard,is_critical_payment_delay,is_low_spender,high_support_risk,is_passive_user,is_low_usage,tenure_segment
0,47,38.0,12.0,10,8,0,250.0,22.0,1,False,False,True,0,1,1,1,0,4
1,39,2.0,27.0,10,24,0,348.0,1.0,1,False,False,False,1,1,1,0,0,1
2,60,21.0,19.0,7,11,0,718.0,16.0,1,True,True,False,0,0,1,1,0,3
3,22,35.0,25.0,8,22,0,805.0,8.0,1,False,True,False,1,0,1,0,0,4
4,19,14.0,16.0,6,14,0,776.0,9.0,1,False,False,True,0,0,1,0,0,3


In [2]:
query = "SELECT * FROM engineered_churn_data"

try:
    logger.info("Attempting to fetch data from PostgreSQL...")
    df = pd.read_sql(query, engine)
    
    logger.info(f"Data ingestion successful! Loaded {df.shape[0]} rows and {df.shape[1]} columns.")
except Exception as e:
    logger.error(f"Failed to ingest data. Error: {str(e)}")
    raise e

[2026-05-13 20:37:40,967: INFO: 1955847757: Attempting to fetch data from PostgreSQL...]
[2026-05-13 20:37:43,378: INFO: 1955847757: Data ingestion successful! Loaded 440832 rows and 18 columns.]


In [3]:
from sklearn.model_selection import train_test_split  
X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
logger.info(f"Split complete. Training: {X_train.shape[0]} rows, Validation: {X_val.shape[0]} rows.")

[2026-05-13 20:37:50,702: INFO: 2529232317: Split complete. Training: 352665 rows, Validation: 88167 rows.]


In [4]:
import mlflow
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import accuracy_score

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("Final_Model_Comparison")

c:\Users\ASUS\customer-churn-mlops-pipeline\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<Experiment: artifact_location='file:c:/Users/ASUS/customer-churn-mlops-pipeline/notebooks/mlruns/2', creation_time=1778566300468, experiment_id='2', last_update_time=1778566300468, lifecycle_stage='active', name='Final_Model_Comparison', tags={}, workspace='default'>

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import yaml

with open("../config/hyperparams.yaml", "r") as f:
    config = yaml.safe_load(f)
    lr_params = config['logistic_regression']

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

with mlflow.start_run(run_name="Logistic_Regression"):
    lr_model = LogisticRegression(**lr_params)
    lr_model.fit(X_train_scaled, y_train)
    
    train_acc = accuracy_score(y_train, lr_model.predict(X_train_scaled))
    val_acc = accuracy_score(y_val, lr_model.predict(X_val_scaled))
    
    mlflow.log_metric("train_accuracy", train_acc)
    mlflow.log_metric("val_accuracy", val_acc)
    
    mlflow.sklearn.log_model(lr_model, name="model")
    
    logger.info(f"Logistic Regression - Train: {train_acc:.4f}, Val: {val_acc:.4f}")

2026/05/13 20:57:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


[2026-05-13 20:57:58,956: INFO: 10242870: Logistic Regression - Train: 0.9675, Val: 0.9680]


In [6]:
from sklearn.ensemble import RandomForestClassifier

with open("../config/hyperparams.yaml", "r") as f:
    config = yaml.safe_load(f)
    rf_params = config['random_forest']

with mlflow.start_run(run_name="Random_Forest"):
    rf_model = RandomForestClassifier(**rf_params)
    rf_model.fit(X_train, y_train)
    
    train_acc = accuracy_score(y_train, rf_model.predict(X_train))
    val_acc = accuracy_score(y_val, rf_model.predict(X_val))
    
    mlflow.log_metric("train_accuracy", train_acc)
    mlflow.log_metric("val_accuracy", val_acc)
    mlflow.sklearn.log_model(rf_model, name="model")
    logger.info(f"Random Forest - Train Accuracy: {train_acc:.4f}, Validation Accuracy: {val_acc:.4f}")

2026/05/13 20:39:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


[2026-05-13 20:39:09,803: INFO: 2104533924: Random Forest - Train Accuracy: 0.9830, Validation Accuracy: 0.9833]


In [7]:
from xgboost import XGBClassifier

with open("../config/hyperparams.yaml", "r") as f:
    config = yaml.safe_load(f)
    xgb_params = config['xgboost']

with mlflow.start_run(run_name="XGBoost"):
    xgb_model = XGBClassifier(**xgb_params)
    xgb_model.fit(X_train, y_train)
    
    train_acc = accuracy_score(y_train, xgb_model.predict(X_train))
    val_acc = accuracy_score(y_val, xgb_model.predict(X_val))
    
    mlflow.log_metric("train_accuracy", train_acc)
    mlflow.log_metric("val_accuracy", val_acc)
    mlflow.xgboost.log_model(xgb_model, name="model")
    logger.info(f"XGBoost -> Train: {train_acc:.4f}, Val: {val_acc:.4f}")

[2026-05-13 20:39:19,596: INFO: 3450257848: XGBoost -> Train: 0.9875, Val: 0.9878]


In [8]:
from catboost import CatBoostClassifier

with open("../config/hyperparams.yaml", "r") as f: 
    config = yaml.safe_load(f)
    cb_params = config['catboost']

with mlflow.start_run(run_name="CatBoost"):
    cb_model = CatBoostClassifier(**cb_params)
    cb_model.fit(X_train, y_train)
    
    train_acc = accuracy_score(y_train, cb_model.predict(X_train))
    val_acc = accuracy_score(y_val, cb_model.predict(X_val))
    
    mlflow.log_metric("train_accuracy", train_acc)
    mlflow.log_metric("val_accuracy", val_acc)
    mlflow.catboost.log_model(cb_model, name="model")
    
    logger.info(f"CatBoost -> Train: {train_acc:.4f}, Val: {val_acc:.4f}")

[2026-05-13 20:39:38,573: INFO: 3380546383: CatBoost -> Train: 0.9876, Val: 0.9879]


In [9]:
with mlflow.start_run(run_name="Final_Best_Model_CatBoost") as run:
    
    mlflow.catboost.log_model(cb_model, name="model")    
    mlflow.set_tag("model_type", "best_performer")
    mlflow.log_metrics({"final_accuracy": 0.9879}) 
    
    run_id = run.info.run_id
    print(f"The best model has been registered. Run ID: {run_id}")

The best model has been registered. Run ID: 1e4b5d94aab24471bd84321d62b60c9e


In [10]:
models_dir = "../models"
os.makedirs(models_dir, exist_ok=True)

model_path = os.path.join(models_dir, "catboost_churn_model.cbm")
cb_model.save_model(model_path)

print(f"The model is locally saved at: {model_path}")

The model is locally saved at: ../models\catboost_churn_model.cbm
